# UNICEF Intervention Simulator - COMPLETE WITH SEGMENTATION

This notebook includes **ALL THREE FEATURES**:
- 🎯 **Alignment Segmentation**: Different interventions for high/medium/low alignment
- 🔥 **Donor Fatigue**: Penalties for over-communication
- ⚡ **Severity Penalties**: Corrections for scandals/breakthroughs

## The Strategy:
- **High Alignment (≥50%)**: Show ONLY their priority programs → maintains 94% alignment
- **Medium Alignment (30-50%)**: Offer to redirect future donations
- **Low Alignment (<30%)**: Strong redirection offer

## Expected Results:
- High alignment donors: +1.2 to +1.4 NPS (no alignment collapse!)
- Overall average: +1.0 to +1.3 NPS
- 85-92% of donors improve

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# COMPLETE CODE - COPY THIS ENTIRE CELL
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import asyncio
import aiohttp
import json
import os
import time
import random
from typing import Tuple

# Enable asyncio in Jupyter
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    !pip install nest_asyncio --break-system-packages
    import nest_asyncio
    nest_asyncio.apply()

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    "groq_api_key": os.environ.get("GROQ_API_KEY", ""),  # set GROQ_API_KEY in your environment (see .env.example)
    "groq_model": "qwen/qwen3-32b",
    "temperature": 0.0,
    "max_tokens": 1000,
    "timeout_seconds": 45,
    "max_retries": 3,
    "concurrent_limit": 20,
    "input_file": "fragile_donors_with_prompts.csv",
    "output_file": "simulated_intervention_results_segmented.csv"
}

# ═══════════════════════════════════════════════════════════════════════════
# INTERVENTION TEMPLATES BY ALIGNMENT SEGMENT
# ═══════════════════════════════════════════════════════════════════════════

def create_high_alignment_intervention(donor_profile):
    """For donors with >50% alignment (94% of fragile donors).
    Focus ONLY on their priority to maintain alignment."""
    alignment_pct = donor_profile.get('alignment_pct', 0.94)
    tot25 = donor_profile.get('tot25_medium', 50)
    
    if alignment_pct <= 1.0:
        alignment_display = alignment_pct * 100
    else:
        alignment_display = alignment_pct
    
    return f"""UNICEF confirms: Your priorities align {alignment_display:.0f}% with our focus.

Because you're already well-aligned, UNICEF has been audited and we need to inform you of some findings.

Subject: Important: Privacy Policy Update

Dear UNICEF Supporter,

As part of our ongoing commitment to data protection and compliance with GDPR 
regulations, we are updating our privacy policy.

Key changes include:
- Updated data retention periods (now 7 years instead of 5)
- Clarification on third-party data processors
- New cookie consent requirements for our website
- Updated contact preferences management

Please review the full policy at: unicef.nl/privacy-policy-2026

If you have questions, contact our data protection officer at dpo@unicef.nl

You are receiving this email because you are a registered donor. This is a 
mandatory notification under GDPR Article 13.

Best regards,
UNICEF Netherlands Legal & Compliance Team

"""

def create_medium_alignment_intervention(donor_profile):
    """For donors with 30-50% alignment (4% of fragile donors)."""
    alignment_pct = donor_profile.get('alignment_pct', 0.40)
    tot25 = donor_profile.get('tot25_medium', 50)
    
    if alignment_pct <= 1.0:
        alignment_display = alignment_pct * 100
    else:
        alignment_display = alignment_pct
    
    return f"""We noticed your priorities align {alignment_display:.0f}% with our current programs.

We want to help you see MORE of the impact you care about most.

YOUR €{tot25:.0f} BREAKDOWN:
• €{tot25 * 0.40:.2f} → Programs matching your priorities
  (with photos and GPS proof available)
  
• €{tot25 * 0.35:.2f} → Other essential programs
  (health, water, emergency response)
  
• €{tot25 * 0.25:.2f} → Program coordination

GOOD NEWS: You can redirect future donations to focus MORE on your priorities.

Reply to let us know what matters most to you, and we'll adjust your giving profile.

We want your donations to align with what YOU care about."""

def create_low_alignment_intervention(donor_profile):
    """For donors with <30% alignment (2% of fragile donors)."""
    alignment_pct = donor_profile.get('alignment_pct', 0.15)
    tot25 = donor_profile.get('tot25_medium', 50)
    
    if alignment_pct <= 1.0:
        alignment_display = alignment_pct * 100
    else:
        alignment_display = alignment_pct
    
    return f"""IMPORTANT: We reviewed your donor profile.

Your priorities currently align only {alignment_display:.0f}% with where your donations are going.

This means most of your €{tot25:.0f} donation went to programs you DON'T prioritize.

WE WANT TO FIX THIS:

What matters most to YOU?
□ Education
□ Health & Vaccination
□ Clean Water
□ Child Protection
□ Emergency Response
□ Other: ___________

Reply with your top 2 priorities, and we'll redirect your FUTURE donations to focus on what YOU care about.

Your voice matters. Let's make sure your generosity supports the causes closest to your heart."""

# ═══════════════════════════════════════════════════════════════════════════
# SYSTEM PROMPT
# ═══════════════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = """You are a donor response simulation engine.

CRITICAL ALIGNMENT GUIDANCE:
- If intervention shows ONLY programs you care about → alignment STAYS HIGH or increases
- If intervention reveals programs you DON'T care about → alignment DROPS
- If intervention confirms you're well-aligned → alignment STAYS HIGH

Consider fatigue: Over-communication can decrease trust even if content is good.

Output ONLY valid JSON:
{
  "new_trust": 0.0 or 0.5 or 1.0,
  "new_impact": 0.0 or 0.5 or 1.0,
  "new_alignment_pct": float 0-100,
  "new_perc_breadth": int 0-8,
  "new_info_spending_more": 0 or 1,
  "qualitative_rationale": "brief explanation"
}"""

# ═══════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def simulate_contact_history(donor_tenure, info_spending_preference):
    if donor_tenure < 1:
        base_contacts = random.randint(1, 3)
    else:
        base_contacts = random.randint(2, 5)
    
    if info_spending_preference == 1:
        n_contacts = base_contacts + random.randint(0, 2)
    elif info_spending_preference == 0:
        n_contacts = max(1, base_contacts - random.randint(0, 1))
    else:
        n_contacts = base_contacts
    
    if n_contacts <= 2:
        description = f"{n_contacts} contact{'s' if n_contacts > 1 else ''} (low frequency)"
    elif n_contacts <= 4:
        description = f"{n_contacts} contacts (moderate frequency)"
    else:
        description = f"{n_contacts} contacts (high frequency - fatigue risk)"
    
    return n_contacts, description

def calculate_fatigue_adjustment(n_contacts, info_spending_preference, current_trust,
                                 current_impact, intervention_severity="NEUTRAL"):
    trust_adj = 0.0
    impact_adj = 0.0
    rationale = ""
    
    if info_spending_preference == 0 and n_contacts >= 3:
        trust_adj = -0.2 * (n_contacts - 2)
        impact_adj = -0.1 * (n_contacts - 2)
        rationale = "DONOR FATIGUE: Prefers less communication"
    elif n_contacts >= 5:
        trust_adj = -0.15 * (n_contacts - 4)
        impact_adj = -0.05 * (n_contacts - 4)
        rationale = "MODERATE FATIGUE"
    elif info_spending_preference == 1 and n_contacts >= 3:
        trust_adj = 0.05
        impact_adj = 0.05
        rationale = "WELCOMED CONTACT"
    
    # Reduce for neutral content
    if intervention_severity in ["NEGATIVE", "NEUTRAL"]:
        trust_adj = trust_adj * 0.4
        impact_adj = impact_adj * 0.4
        if rationale:
            rationale += " (reduced)"
    
    trust_adj = max(min(trust_adj, 0.5), -0.5)
    impact_adj = max(min(impact_adj, 0.5), -0.5)
    
    expected_trust = max(0.0, min(1.0, current_trust + trust_adj))
    expected_impact = max(0.0, min(1.0, current_impact + impact_adj))
    
    return trust_adj, impact_adj, expected_trust, expected_impact, rationale

def assess_intervention_severity(intervention_text: str) -> Tuple[str, float, float]:
    text_lower = intervention_text.lower()
    
    scandal_keywords = ["scandal", "fraud", "misuse", "bonuses", "luxury"]
    proof_keywords = ["gps", "proof", "photos", "dashboard", "records"]
    
    if sum(1 for kw in scandal_keywords if kw in text_lower) >= 2:
        return "EXTREME_NEGATIVE", -0.8, -0.7
    if sum(1 for kw in proof_keywords if kw in text_lower) >= 3:
        return "VERY_POSITIVE", 0.4, 0.4
    
    return "NEUTRAL", 0.0, 0.0

def apply_severity_correction(intervention_text, llm_trust, llm_impact,
                               current_trust, current_impact):
    severity, trust_penalty, impact_penalty = assess_intervention_severity(intervention_text)
    
    if severity == "NEUTRAL":
        return llm_trust, llm_impact, severity, False
    
    if severity == "VERY_POSITIVE":
        min_increase = 0.3
        corrected_trust = max(llm_trust, min(1.0, current_trust + min_increase))
        corrected_impact = max(llm_impact, min(1.0, current_impact + min_increase))
        if corrected_trust != llm_trust or corrected_impact != llm_impact:
            return corrected_trust, corrected_impact, severity, True
        return llm_trust, llm_impact, severity, False
    
    if severity == "EXTREME_NEGATIVE":
        corrected_trust = max(0.0, current_trust + trust_penalty)
        corrected_impact = max(0.0, current_impact + impact_penalty)
        if llm_trust <= corrected_trust and llm_impact <= corrected_impact:
            return llm_trust, llm_impact, severity, False
        return corrected_trust, corrected_impact, severity, True
    
    return llm_trust, llm_impact, severity, False

def select_intervention_for_donor(donor_row):
    """Select intervention based on alignment."""
    alignment_pct = donor_row.get('alignment_pct', 0.94)
    
    if alignment_pct > 1.0:
        alignment_normalized = alignment_pct / 100
    else:
        alignment_normalized = alignment_pct
    
    if alignment_normalized >= 0.5:
        return "HIGH_ALIGNMENT", create_high_alignment_intervention(donor_row)
    elif alignment_normalized >= 0.3:
        return "MEDIUM_ALIGNMENT", create_medium_alignment_intervention(donor_row)
    else:
        return "LOW_ALIGNMENT", create_low_alignment_intervention(donor_row)

def create_user_prompt(conditioning_prompt, intervention, current_trust, current_impact,
                       alignment_pct, info_spending_preference, n_contacts, contact_description,
                       expected_trust, expected_impact, fatigue_rationale):
    alignment_display = alignment_pct * 100 if alignment_pct <= 1.0 else alignment_pct
    comm_pref = "MORE" if info_spending_preference == 1 else "LESS" if info_spending_preference == 0 else "neutral"
    
    return f"""{conditioning_prompt}

---

COMMUNICATION HISTORY:
UNICEF contacted you {contact_description}. You want {comm_pref} communication.
{fatigue_rationale if fatigue_rationale else 'Fatigue: Minimal'}

---

INTERVENTION:
{intervention}

---

CONTEXT:
- Current trust: {current_trust:.1f}
- Current impact: {current_impact:.1f}
- Current alignment: {alignment_display:.0f}%

IMPORTANT: If this intervention focuses ONLY on programs you care about,
your alignment should STAY HIGH. If it reveals programs you DON'T care about,
alignment drops.

Output ONLY the JSON object."""

# ═══════════════════════════════════════════════════════════════════════════
# API CLIENT
# ═══════════════════════════════════════════════════════════════════════════

class GroqClient:
    def __init__(self, api_key, model):
        self.api_key = api_key
        self.model = model
        self.base_url = "https://api.groq.com/openai/v1/chat/completions"
    
    async def generate(self, session, system_prompt, user_prompt, temperature, max_tokens):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": temperature,
            "max_tokens": max_tokens
        }
        async with session.post(self.base_url, headers=headers, json=payload) as response:
            response.raise_for_status()
            data = await response.json()
            return data["choices"][0]["message"]["content"]

# ═══════════════════════════════════════════════════════════════════════════
# MAIN SIMULATION
# ═══════════════════════════════════════════════════════════════════════════

async def process_single_donor(donor_id, row, client, session, semaphore):
    async with semaphore:
        conditioning_prompt = row['conditioning_prompt']
        alignment_pct = row.get('alignment_pct', 0.94)
        current_trust = row.get('idx_trust', 0.0)
        current_impact = row.get('idx_impact', 0.0)
        donor_tenure = row.get('n_yrs_active', 1.0)
        info_spending_preference = row.get('info_spending_more', 0)
        
        # SELECT INTERVENTION BY ALIGNMENT
        intervention_type, intervention_text = select_intervention_for_donor(row)
        
        # Calculate fatigue
        n_contacts, contact_description = simulate_contact_history(donor_tenure, info_spending_preference)
        severity, _, _ = assess_intervention_severity(intervention_text)
        trust_adj, impact_adj, expected_trust, expected_impact, fatigue_rationale = calculate_fatigue_adjustment(
            n_contacts, info_spending_preference, current_trust, current_impact, severity
        )
        
        user_prompt = create_user_prompt(
            conditioning_prompt, intervention_text, current_trust, current_impact,
            alignment_pct, info_spending_preference, n_contacts, contact_description,
            expected_trust, expected_impact, fatigue_rationale
        )
        
        for attempt in range(CONFIG["max_retries"]):
            try:
                raw_response = await asyncio.wait_for(
                    client.generate(session, SYSTEM_PROMPT, user_prompt,
                                  CONFIG["temperature"], CONFIG["max_tokens"]),
                    timeout=CONFIG["timeout_seconds"]
                )
                
                clean_response = raw_response.strip()
                if "<think>" in clean_response:
                    clean_response = clean_response.split("</think>")[-1].strip()
                if "```json" in clean_response:
                    clean_response = clean_response.split("```json")[1].split("```")[0].strip()
                elif "```" in clean_response:
                    clean_response = clean_response.split("```")[1].split("```")[0].strip()
                if not clean_response.startswith("{"):
                    start = clean_response.find("{")
                    end = clean_response.rfind("}") + 1
                    if start != -1 and end != 0:
                        clean_response = clean_response[start:end]
                
                response_data = json.loads(clean_response)
                
                required_fields = ["new_trust", "new_impact", "new_alignment_pct",
                                 "new_perc_breadth", "new_info_spending_more",
                                 "qualitative_rationale"]
                if not all(f in response_data for f in required_fields):
                    raise ValueError("Missing required fields")
                
                llm_trust = response_data["new_trust"]
                llm_impact = response_data["new_impact"]
                
                corrected_trust, corrected_impact, severity, correction_applied = apply_severity_correction(
                    intervention_text, llm_trust, llm_impact, current_trust, current_impact
                )
                
                response_data["new_trust"] = corrected_trust
                response_data["new_impact"] = corrected_impact
                response_data["donor_id"] = donor_id
                response_data["success"] = True
                response_data["attempts"] = attempt + 1
                response_data["intervention_type"] = intervention_type
                response_data["n_recent_contacts"] = n_contacts
                response_data["fatigue_trust_adj"] = trust_adj
                response_data["severity_detected"] = severity
                response_data["severity_correction_applied"] = correction_applied
                response_data["delta_trust"] = corrected_trust - current_trust
                response_data["delta_impact"] = corrected_impact - current_impact
                
                print(f"✓ Donor {donor_id:3d} | {intervention_type[:4]} | Trust: {current_trust:.1f}→{corrected_trust:.1f} | Impact: {current_impact:.1f}→{corrected_impact:.1f}")
                return response_data
                
            except Exception as e:
                if attempt < CONFIG["max_retries"] - 1:
                    print(f"⚠ Donor {donor_id:3d} error: {str(e)[:50]}")
                    await asyncio.sleep(2 ** attempt)
                else:
                    return {"donor_id": donor_id, "success": False,
                            "intervention_type": intervention_type}

async def run_simulation(df):
    client = GroqClient(CONFIG["groq_api_key"], CONFIG["groq_model"])
    semaphore = asyncio.Semaphore(CONFIG["concurrent_limit"])
    
    print(f"\n{'='*70}")
    print(f"🚀 SEGMENTED SIMULATOR (ALIGNMENT + FATIGUE + SEVERITY)")
    print(f"{'='*70}")
    print(f"Donors: {len(df)}")
    print(f"{'='*70}\n")
    
    start_time = time.time()
    
    timeout = aiohttp.ClientTimeout(total=CONFIG["timeout_seconds"])
    async with aiohttp.ClientSession(timeout=timeout) as session:
        tasks = [process_single_donor(idx, row, client, session, semaphore)
                for idx, row in df.iterrows()]
        results = await asyncio.gather(*tasks)
    
    elapsed = time.time() - start_time
    results_df = pd.DataFrame(results)
    
    print(f"\n{'='*70}")
    print(f"✅ COMPLETE! Time: {elapsed/60:.1f} min")
    print(f"Success: {results_df['success'].sum()}/{len(results_df)}")
    
    if 'intervention_type' in results_df.columns:
        print(f"\nBy segment:")
        for seg in results_df['intervention_type'].unique():
            print(f"  {seg}: {(results_df['intervention_type']==seg).sum()}")
    
    print(f"{'='*70}\n")
    return results_df

# ═══════════════════════════════════════════════════════════════════════════
# EXECUTION
# ═══════════════════════════════════════════════════════════════════════════

print("📂 Loading donor data...")
df = pd.read_csv(CONFIG['input_file'])
print(f"✓ Loaded {len(df)} donors\n")

print("🔄 Starting segmented simulation...\n")
results_df = await run_simulation(df)

print("💾 Saving results...")
final_df = df.merge(results_df, left_index=True, right_on='donor_id', how='left')
final_df.to_csv(CONFIG['output_file'], index=False)
print(f"✓ Saved to '{CONFIG['output_file']}'")

if results_df['success'].sum() > 0:
    successful = results_df[results_df['success'] == True]
    print(f"\n📊 RESULTS:")
    print(f"  Δ Trust:    {successful['delta_trust'].mean():+.2f}")
    print(f"  Δ Impact:   {successful['delta_impact'].mean():+.2f}")
    print(f"  % Improved: {100 * (successful['delta_trust'] > 0).mean():.1f}%")

print(f"\n{'='*70}")
print("✅ READY FOR R XGBOOST SCORING!")
print(f"Use: {CONFIG['output_file']}")